In [2]:
pip install ipynb

Note: you may need to restart the kernel to use updated packages.


In [1]:
import tkinter as tk
from tkinter import messagebox, ttk
from ipynb.fs.full.data_manager import get_jobs, save_application, get_applications


class EmployeeDashboard(tk.Tk):

    def __init__(self, current_user):
        super().__init__()

        # User session data passed from login: {'id': '1', 'name': 'John Doe', 'role': 'Employee'}
        self.current_user = current_user

        self.title("WorkLink - Job Seeker Dashboard")
        self.geometry("900x550")
        self.configure(bg="#f5f6fa")

        # Top Navigation/Welcome Bar
        self.top_bar = tk.Frame(self, bg="#2c3e50", height=60)
        self.top_bar.pack(fill="x", side="top")
        self.top_bar.pack_propagate(False)

        welcome_lbl = tk.Label(
            self.top_bar,
            text=f"Welcome back, {self.current_user['name']}!",
            font=("Arial", 14, "bold"),
            fg="white",
            bg="#2c3e50",
        )
        welcome_lbl.pack(side="left", padx=20)

        # Main Workspace Frame (Dynamic Content)
        self.main_container = tk.Frame(self, bg="#f5f6fa")
        self.main_container.pack(fill="both", expand=True, padx=20, pady=20)

        # Show the primary dashboard home on start
        self.show_dashboard_home()

    def clear_container(self):
        """Clears the workspace frame before rendering a new screen."""
        for widget in self.main_container.winfo_children():
            widget.destroy()

    def show_dashboard_home(self):
        """Step 4 Action Menu - Employee home panel."""
        self.clear_container()

        # Grid config for layout
        self.main_container.columnconfigure(0, weight=1)
        self.main_container.columnconfigure(1, weight=1)

        # Title Block
        title_lbl = tk.Label(
            self.main_container,
            text="Job Seeker Control Panel",
            font=("Arial", 20, "bold"),
            bg="#f5f6fa",
            fg="#2c3e50",
        )
        title_lbl.grid(row=0, column=0, columnspan=2, pady=(0, 30), sticky="w")

        # Action Button 1: Find Jobs
        browse_btn = tk.Button(
            self.main_container,
            text="🔍 Find Jobs\nBrowse listings & apply",
            font=("Arial", 12, "bold"),
            bg="#3498db",
            fg="white",
            bd=0,
            cursor="hand2",
            padx=20,
            pady=20,
            command=self.show_browse_jobs,
        )
        browse_btn.grid(row=1, column=0, padx=20, pady=10, sticky="nsew")

        # Action Button 2: View Applied Jobs
        history_btn = tk.Button(
            self.main_container,
            text="📁 Applied Jobs\nTrack your applications",
            font=("Arial", 12, "bold"),
            bg="#2ecc71",
            fg="white",
            bd=0,
            cursor="hand2",
            padx=20,
            pady=20,
            command=self.show_applied_history,
        )
        history_btn.grid(row=1, column=1, padx=20, pady=10, sticky="nsew")

    def show_browse_jobs(self):
        """Browse Jobs Screen featuring filtering capabilities and a structured Treeview Table."""
        self.clear_container()

        # Control Bar for Sub-navigation & Filters
        nav_frame = tk.Frame(self.main_container, bg="#f5f6fa")
        nav_frame.pack(fill="x", pady=(0, 15))

        back_btn = tk.Button(
            nav_frame,
            text="← Back to Dashboard",
            font=("Arial", 10, "bold"),
            command=self.show_dashboard_home,
            bg="#7f8c8d",
            fg="white",
            bd=0,
            padx=10,
        )
        back_btn.pack(side="left")

        # Optional Feature: "No Experience Required" Entry-level Filter
        self.filter_var = tk.BooleanVar(value=False)
        filter_chk = tk.Checkbutton(
            nav_frame,
            text="Show 'No Experience Required' Only",
            variable=self.filter_var,
            command=self.load_jobs_into_table,
            font=("Arial", 10),
            bg="#f5f6fa",
            activebackground="#f5f6fa",
        )
        filter_chk.pack(side="right", padx=10)

        # --- THE TREEVIEW JOB TABLE ---
        table_frame = tk.Frame(self.main_container)
        table_frame.pack(fill="both", expand=True)

        # Scrollbar setup
        tree_scroll = tk.Scrollbar(table_frame)
        tree_scroll.pack(side="right", fill="y")

        # Define table columns matching data_manager schemas
        columns = ("id", "title", "skills", "experience", "description")
        self.job_table = ttk.Treeview(
            table_frame,
            columns=columns,
            show="headings",
            yscrollcommand=tree_scroll.set,
        )
        tree_scroll.config(command=self.job_table.yview)

        # Style layout headers
        self.job_table.heading("id", text="Job ID")
        self.job_table.heading("title", text="Job Title")
        self.job_table.heading("skills", text="Required Skills")
        self.job_table.heading("experience", text="Experience Level")
        self.job_table.heading("description", text="Description")

        # Format column dimension spacing
        self.job_table.column("id", width=60, anchor="center")
        self.job_table.column("title", width=180, anchor="w")
        self.job_table.column("skills", width=160, anchor="w")
        self.job_table.column("experience", width=140, anchor="center")
        self.job_table.column("description", width=260, anchor="w")

        self.job_table.pack(fill="both", expand=True)

        # --- APPLICATION TRIGGER BAR ---
        action_bar = tk.Frame(self.main_container, bg="#f5f6fa")
        action_bar.pack(fill="x", pady=15)

        apply_btn = tk.Button(
            action_bar,
            text="🚀 Apply for Selected Job",
            font=("Arial", 11, "bold"),
            bg="#e67e22",
            fg="white",
            bd=0,
            padx=20,
            pady=8,
            command=self.submit_application,
        )
        apply_btn.pack(side="right")

        # Fetch and load rows right away
        self.load_jobs_into_table()

    def load_jobs_into_table(self):
        """Fetches data using your get_jobs system tool and populates the table."""
        # Wipe structural visual entries first
        for item in self.job_table.get_children():
            self.job_table.delete(item)

        try:
            # Step 4: Core functional integration
            all_jobs = get_jobs()
        except NameError:
            # Fallback mock data if executed standalone without your core data_manager cell file assets
            all_jobs = [
                {
                    "id": "1",
                    "title": "Junior Python Developer",
                    "skills": "Python, SQL",
                    "experience": "No experience required",
                    "description": "Build desktop dashboards.",
                },
                {
                    "id": "2",
                    "title": "Data Analyst Intern",
                    "skills": "Excel, Pandas",
                    "experience": "Entry-level",
                    "description": "Clean and format structural platform records.",
                },
                {
                    "id": "3",
                    "title": "Senior Rust Engineer",
                    "skills": "Rust, Systems Programming",
                    "experience": "5+ Years",
                    "description": "High throughput memory optimization architectures.",
                },
            ]

        # Populate rows with selective structural parameters applied
        for job in all_jobs:
            # Process 'No Experience Required' Filter toggle state
            if self.filter_var.get():
                exp_lower = job.get("experience", "").lower()
                if (
                    "no experience" not in exp_lower
                    and "entry" not in exp_lower
                ):
                    continue  # Skip higher baseline records

            self.job_table.insert(
                "",
                "end",
                values=(
                    job.get("id"),
                    job.get("title"),
                    job.get("skills"),
                    job.get("experience"),
                    job.get("description"),
                ),
            )

    def submit_application(self):
        """Extracts tracking parameters dynamically from selected Treeview rows and commits data."""
        selected_item = self.job_table.selection()

        if not selected_item:
            messagebox.showwarning(
                "Selection Missing",
                "Please choose a job opportunity from the table list first.",
            )
            return

        # Acquire array fields belonging to target highlight entry
        row_values = self.job_table.item(selected_item, "values")
        selected_job_id = row_values[0]
        selected_title = row_values[1]

        # Call the requested notebook function: save_application(job_id, employee_id)
        try:
            success = save_application(selected_job_id, self.current_user["id"])
        except NameError:
            # Sample system state handling simulation logic
            success = True

        if success:
            messagebox.showinfo(
                "Application Successful",
                f"Your application for '{selected_title}' has been registered in the system!",
            )
        else:
            messagebox.showerror(
                "Duplicate Entry",
                f"You have already applied for the position: '{selected_title}'.",
            )

    def show_applied_history(self):
        """Secondary helper viewing window rendering historical application statuses."""
        self.clear_container()

        nav_frame = tk.Frame(self.main_container, bg="#f5f6fa")
        nav_frame.pack(fill="x", pady=(0, 15))

        back_btn = tk.Button(
            nav_frame,
            text="← Back to Dashboard",
            font=("Arial", 10, "bold"),
            command=self.show_dashboard_home,
            bg="#7f8c8d",
            fg="white",
            bd=0,
            padx=10,
        )
        back_btn.pack(side="left")

        lbl = tk.Label(
            self.main_container,
            text="Historical Job Track Record",
            font=("Arial", 14, "bold"),
            bg="#f5f6fa",
            fg="#2c3e50",
        )
        lbl.pack(pady=10, anchor="w")

        history_table = ttk.Treeview(
            self.main_container, columns=("job_id"), show="headings"
        )
        history_table.heading("job_id", text="Applied Job ID References")
        history_table.pack(fill="both", expand=True)

        try:
            user_apps = get_applications(self.current_user["id"])
            for app in user_apps:
                history_table.insert("", "end", values=(app.get("job_id"),))
        except NameError:
            history_table.insert("", "end", values=("Sample Application #1",))


# --- RUN WINDOW ENVIRONMENT DEMO ---
if __name__ == "__main__":
    # Simulated Session environment variable populated by step 2 setup screen sequence:
    mock_active_employee = {
        "id": "42",
        "name": "Alex Mercer",
        "role": "Job Seeker",
    }

    app = EmployeeDashboard(current_user=mock_active_employee)
    app.mainloop()